# 第13章 RAG (RAG搭載モデルを検証)

## 13.3 RAG 向けに LLM を指示チューニングする

### 13.3.2 指示チューニングしたモデルを LangChain で使う

#### 環境の準備

In [1]:
!pip install 'datasets<4.0.0' transformers[torch,sentencepiece] langchain langchain-community langchain-huggingface faiss-cpu jq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 129.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.8/773.8 kB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.3
    Uninstalling langchain-core-1.4.3:
      Successfully uninstalled langchain-core-1.4.3
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling 

In [2]:
from transformers.trainer_utils import set_seed
from google.colab import drive

set_seed(42)
drive.mount("drive")

Mounted at drive


#### Chat Modelの作成

In [3]:
import torch
from langchain_huggingface import (
    ChatHuggingFace,
    HuggingFacePipeline,
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
)

# Hugging Face Hubにおけるモデル名を指定
model_name = "llm-book/Swallow-7b-hf-oasst1-21k-ja-aio-retriever"

# モデルを読み込む
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# トークナイザを読み込む
tokenizer = AutoTokenizer.from_pretrained(model_name)

# テキスト生成用のパラメータを指定
generation_config = {
    "max_new_tokens": 32,
    "do_sample": False,
    "temperature": None,
    "top_p": None,
}

# テキスト生成を行うパイプラインを作成
text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    **generation_config,
)

# パイプラインからLangChainのLLMコンポーネントを作成
llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

# LLMコンポーネントを元にChat Modelコンポーネントを作成
chat_model = ChatHuggingFace(llm=llm, tokenizer=tokenizer)

config.json:   0%|          | 0.00/770 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/914k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.30M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


#### Embedding Modelの作成

In [4]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

# Hugging Face Hubにおけるモデル名を指定
embedding_model_name = "BAAI/bge-m3"

# モデル名からEmbedding Modelを初期化
embedding_model = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={"model_kwargs": {"torch_dtype": torch.float16}},
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

#### データストアの構築

In [5]:
from datasets import load_dataset
ds = load_dataset(
    "singletongue/wikipedia-utils",
    "passages-c400-jawiki-20240401",
)

README.md:   0%|          | 0.00/6.75k [00:00<?, ?B/s]

passages-c400-jawiki-20240401/train-0000(…):   0%|          | 0.00/300M [00:00<?, ?B/s]

passages-c400-jawiki-20240401/train-0000(…):   0%|          | 0.00/283M [00:00<?, ?B/s]

passages-c400-jawiki-20240401/train-0000(…):   0%|          | 0.00/260M [00:00<?, ?B/s]

passages-c400-jawiki-20240401/train-0000(…):   0%|          | 0.00/252M [00:00<?, ?B/s]

...

passages-c400-jawiki-20240401/train-0000(…):   0%|          | 0.00/240M [00:00<?, ?B/s]

passages-c400-jawiki-20240401/train-0000(…):   0%|          | 0.00/238M [00:00<?, ?B/s]

passages-c400-jawiki-20240401/train-0000(…):   0%|          | 0.00/237M [00:00<?, ?B/s]

passages-c400-jawiki-20240401/train-0000(…):   0%|          | 0.00/235M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5807053 [00:00<?, ? examples/s]

In [6]:
print(ds["train"])
print(ds["train"].column_names)
print(ds["train"][0])

Dataset({
    features: ['id', 'pageid', 'revid', 'title', 'section', 'text'],
    num_rows: 5807053
})
['id', 'pageid', 'revid', 'title', 'section', 'text']
{'id': 1, 'pageid': 5, 'revid': 99347164, 'title': 'アンパサンド', 'section': '__LEAD__', 'text': 'アンパサンド(&, 英語: ampersand)は、並立助詞「...と...」を意味する記号である。ラテン語で「...と...」を表す接続詞 "et" の合字を起源とする。現代のフォントでも、Trebuchet MS など一部のフォントでは、"et" の合字であることが容易にわかる字形を使用している。'}


In [10]:
train = ds["train"].select(range(200_000))  # 件数は環境に合わせて調整
split_documents = [
    Document(page_content=text, metadata={"title": title, "section": section})
    for text, title, section in zip(train["text"], train["title"], train["section"])
]
print(len(split_documents))

200000


#### 検索対象の文書のベクトルインデックスの作成

In [11]:
from langchain_community.vectorstores import FAISS

# 分割後の文書と文埋め込みモデルを用いて、Faissのベクトルインデックスを作成
vectorstore = FAISS.from_documents(split_documents, embedding_model)

# ベクトルインデックスに登録された文書数を確認
print(vectorstore.index.ntotal)

200000


#### Retrieverコンポーネントの作成

In [12]:
# ベクトルインデックスを元に文書の検索を行うRetrieverを初期化
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [13]:
# 動作確認
test_retriever_doc = retriever.invoke("日本の神は？")
print(test_retriever_doc[0].page_content)

神は、自然を感じ取り、そのもののままでは厳しい自然の中で、人間として文化的な生活を営むのにふさわしい環境と状態を、自然との調和に配慮しながらバランスを取り調節していき、人民生活を見回って、生活するための知恵や知識のヒントを与えたり、少し手伝ってあげたり、体や物を借りたときや何かやってもらったときなどには少しお礼をしたり、それが、日本の「神(かみ)」が行っていた仕事のひとつであり、日本人にとって「神」は、とても身近な存在であった。また、神道における神は、理念的・抽象的存在ではなく、具体的な現象において観念されるため、自然現象が恵みとともに災害をもたらすのと同様に、神も荒魂・和魂の両面を持ち、人間にとって善悪双方をもたらすものと考えられている。神は、地域社会を守り、現世の人間に恩恵を与える穏やかな「守護神」であるが、天変地異を引き起こし、病や死を招き寄せる「祟る」性格も持っている。


In [14]:
for i in range(3):
    print(test_retriever_doc[i].page_content)

神は、自然を感じ取り、そのもののままでは厳しい自然の中で、人間として文化的な生活を営むのにふさわしい環境と状態を、自然との調和に配慮しながらバランスを取り調節していき、人民生活を見回って、生活するための知恵や知識のヒントを与えたり、少し手伝ってあげたり、体や物を借りたときや何かやってもらったときなどには少しお礼をしたり、それが、日本の「神(かみ)」が行っていた仕事のひとつであり、日本人にとって「神」は、とても身近な存在であった。また、神道における神は、理念的・抽象的存在ではなく、具体的な現象において観念されるため、自然現象が恵みとともに災害をもたらすのと同様に、神も荒魂・和魂の両面を持ち、人間にとって善悪双方をもたらすものと考えられている。神は、地域社会を守り、現世の人間に恩恵を与える穏やかな「守護神」であるが、天変地異を引き起こし、病や死を招き寄せる「祟る」性格も持っている。
ただし、「日本国主」である天照大神であっても、それは六道など仏教的宇宙観の一角としての下界である「日本」という領域に限定される最高神であって、仏教的宇宙観全体の支配者である梵天などの仏よりは下位として見なされる場合があり、起請文にも仏の名前を上段に列記し、下段に天照大御神をはじめとする神々の名が列記される例が見られる。また、中世期には天照大神は大日如来の垂迹として信仰され、仏教信仰と結びつけられた。
天照大神(あまてらすおおかみ)または天照大御神(あまてらすおおみかみ)は、日本神話に主神として登場する神。女神と解釈され、高天原を統べる主宰神で、皇祖神である。『記紀』においては、太陽神の性格と巫女の性格を併せ持つ存在として描かれている。神武天皇は来孫。太陽神、農耕神、機織神など多様な神格を持つ。天岩戸の神隠れで有名な神で、神社としては三重県伊勢市にある伊勢神宮内宮が特に有名。


#### RAGのChainの構築と実行

In [15]:
from langchain_core.prompts import ChatPromptTemplate

# 任意のqueryからメッセージを構築するPrompt Templateを作成
rag_prompt_text = (
    "あなたには今からクイズに答えてもらいます。"
    "問題を与えますので、その解答のみを簡潔に出力してください。\n"
    "また解答の参考になりうるテキストを与えます。"
    "解答を含まない場合もあるのでその場合は無視してください。\n\n"
    "---\n{context}\n---\n\n問題: {query}"
)
rag_prompt_template = ChatPromptTemplate.from_messages(
    [("user", rag_prompt_text)]
)

In [16]:
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

def format_documents_func(documents: list[Document]) -> str:
    """文書のリストを改行で連結した一つの文字列として返す"""
    return "\n\n".join(
        document.page_content for document in documents
    )

# 定義した関数の処理を行うRunnableを作成
format_documents = RunnableLambda(format_documents_func)

In [17]:
from langchain_core.prompt_values import ChatPromptValue

def chat_model_resp_only_func(
    chat_prompt_value: ChatPromptValue,
) -> str:
    """chat_modelにchat_prompt_valueを入力し、
    出力からモデルの応答部分のみを文字列で返す"""
    chat_prompt = chat_model._to_chat_prompt(
        chat_prompt_value.messages
    )
    chat_output_message = chat_model.invoke(chat_prompt_value)
    response_text = chat_output_message.content[len(chat_prompt) :]
    return response_text

# 定義した関数の処理を行うRunnableを作成
chat_model_resp_only = RunnableLambda(chat_model_resp_only_func)

In [18]:
from langchain_core.runnables import RunnablePassthrough

# RAGの一連の処理を行うChainを作成
rag_chain = (
    {
        "context": retriever | format_documents,
        "query": RunnablePassthrough(),
    }
    | rag_prompt_template
    | chat_model_resp_only
)

### (EXP) AI王データセットを使用して、検証

#### データセットの準備


In [19]:
from datasets import load_dataset
# Hugging Face Hubのllm-book/aio-retrieverのリポジトリから
# AI王データセットを読み込む
dataset = load_dataset(
    "llm-book/aio-retriever",
    trust_remote_code=True
)

README.md:   0%|          | 0.00/2.56k [00:00<?, ?B/s]

aio-retriever.py:   0%|          | 0.00/3.58k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/22335 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [20]:
# 読み込まれたデータセットの形式と事例数を確認
print(dataset)
print(dataset["validation"])
print(dataset["validation"][0])


DatasetDict({
    train: Dataset({
        features: ['qid', 'competition', 'timestamp', 'section', 'number', 'original_question', 'original_answer', 'original_additional_info', 'question', 'answers', 'passages', 'positive_passage_indices', 'negative_passage_indices'],
        num_rows: 22335
    })
    validation: Dataset({
        features: ['qid', 'competition', 'timestamp', 'section', 'number', 'original_question', 'original_answer', 'original_additional_info', 'question', 'answers', 'passages', 'positive_passage_indices', 'negative_passage_indices'],
        num_rows: 1000
    })
})
Dataset({
    features: ['qid', 'competition', 'timestamp', 'section', 'number', 'original_question', 'original_answer', 'original_additional_info', 'question', 'answers', 'passages', 'positive_passage_indices', 'negative_passage_indices'],
    num_rows: 1000
})
{'qid': 'AIO02-0001', 'competition': '第2回AI王', 'timestamp': '2021/01/29', 'section': '開発データ問題', 'number': '1', 'original_question': '映画『ウエスト・サ

#### 検証の準備

In [21]:
from datasets import Dataset
from tqdm.notebook import tqdm

def evaluate_rag(
    rag_chain,
    dataset: Dataset,
    limit: int = 100
) -> tuple[list[str], list[list[str]], float]:
    """RAGチェーンを用いてデータセットの各問題に回答し、正解率を算出"""
    pred_answers = []
    gold_answers = []
    num_correct = 0

    # 全件実行すると時間がかかるため、指定した件数(limit)で評価
    for example in tqdm(dataset.select(range(limit))):
        question = example["question"]
        # 正解のリスト（別解が含まれる場合があるためリスト形式）
        gold_answer_list = example["answers"]

        # RAGチェーンに問題を入力し、出力を得る
        pred_answer = rag_chain.invoke(question)  # templateでquestionのみを引数に持つように調整済み

        # モデルの答えが正解リストのいずれかと完全に一致していれば正答とカウント
        if pred_answer in gold_answer_list:
            num_correct += 1

        # モデルの答えと正解リストをそれぞれ追加
        pred_answers.append(pred_answer)
        gold_answers.append(gold_answer_list)

    # 正解率を計算
    accuracy = num_correct / len(pred_answers)

    return pred_answers, gold_answers, accuracy

In [22]:
# 構築したRAGチェーンを使って評価（時間短縮のため100件で検証）
pred_answers, gold_answers, accuracy = evaluate_rag(
    rag_chain, dataset["validation"], limit=100
)

print(f"正解率: {accuracy:.1%}")

  0%|          | 0/100 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentatio

正解率: 54.0%


In [23]:
# 構築したRAGチェーンを使って評価（時間短縮のため100件で検証）
pred_answers, gold_answers, accuracy = evaluate_rag(
    rag_chain, dataset["train"], limit=100
)

print(f"正解率: {accuracy:.1%}")

  0%|          | 0/100 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

正解率: 63.0%


#### 実例を確認

In [26]:
pred_answer = rag_chain.invoke(dataset["train"][0]["question"])
gold_answer = dataset["train"][0]["answers"][0]
print(pred_answer)
print(gold_answer)

[transformers] Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


26文字
26文字


In [28]:
for i in range(10):
    pred_answer = rag_chain.invoke(dataset["train"][i]["question"])
    gold_answer = dataset["train"][i]["answers"][0]
    print(i)
    print(f"pred_answer: {pred_answer}")
    print(f"gold_answer: {gold_answer}")

[transformers] Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


0
pred_answer: 26文字
gold_answer: 26文字


[transformers] Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1
pred_answer: 骨川
gold_answer: 骨川


[transformers] Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

6
pred_answer: 1/36
gold_answer: 6分の1


[transformers] Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


7
pred_answer: チューリップ
gold_answer: オリーブ


[transformers] Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


8
pred_answer: みにくいアヒルの子
gold_answer: みにくいあひるの子
9
pred_answer: バリトン
gold_answer: バリトン


In [35]:
# 明らかに間違っている事例で確認
# 7
# pred_answer: チューリップ
# gold_answer: オリーブ
print("質問文: ", dataset["train"][7]["question"])
for i in range(3):
    print(f"検索文書{i}: ", retriever.invoke(dataset["train"][7]["question"])[i].page_content)

質問文:  「国際連合」の旗に描かれている植物といったら何でしょう?
検索文書0:  国花は蓮、国樹は印度菩提樹、国獣はベンガルトラ、国鳥はインドクジャク、国の遺産動物はインドゾウである。
検索文書1:  - 紅葉 - 落葉 - 彼岸花 - 藤袴 - 桔梗 - ダリア - 萩 - 女郎花 - 芒 - コスモス - 鶏頭 - 金木犀 - 菊 - 竜胆 - 背高泡立草 - 芋 - 瓜 - 糸瓜 - 撫子 - 葛の花 - 朝顔
検索文書2:  国旗は3色で構成され、緑はイスラムの国であること、白は平和と平和愛好、赤は勇気を表している。中央のモチーフは4本の剣と三日月を用いてアラビア語のアッラー(الله)とチューリップを象徴する。アッラーは、イスラムの唯一神で、チューリップはイスラムと祖国のために戦死した殉教者を表す。


In [38]:
# コサイン類似度
import numpy as np
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


In [40]:
query = dataset["train"][7]["question"]
docs = retriever.invoke(query)[:3]

# 埋め込みモデルの取得（retrieverの構成に依存）
# FAISS / Chroma などのVectorStoreRetrieverの場合:
embeddings = retriever.vectorstore.embeddings

# ベクトル化
q_vec = np.array(embeddings.embed_query(query))
doc_vecs = np.array(embeddings.embed_documents([d.page_content for d in docs]))

for i, dv in enumerate(doc_vecs):
    sim = cosine_sim(q_vec, dv)
    print(f"検索文書{i} のコサイン類似度: {sim:.4f}")

検索文書0 のコサイン類似度: 0.5425
検索文書1 のコサイン類似度: 0.5246
検索文書2 のコサイン類似度: 0.5171


In [43]:
# 明らかに間違っている事例で確認
# 4
# pred_answer: 三英傑
# gold_answer: ホトトギス
print("質問文: ", dataset["train"][4]["question"])
for i in range(3):
    print(f"検索文書{i}: ", retriever.invoke(dataset["train"][4]["question"])[i].page_content)

質問文:  織田信長、豊臣秀吉、徳川家康という3人の戦国武将の性格を表現するのに用いられる鳥は何でしょう?
検索文書0:  織田 信長(おだ のぶなが)は、日本の戦国時代から安土桃山時代にかけての武将・大名。戦国の三英傑の一人。尾張国(現在の愛知県)出身。織田信秀の嫡男。家督争いの混乱を収めた後に、桶狭間の戦いで今川義元を討ち取り、勢力を拡大した。足利義昭を奉じて上洛し、後には義昭を追放することで、畿内を中心に独自の中央政権(「織田政権」)を確立して天下人となった戦国時代を代表する英雄である。しかし、天正10年6月2日(1582年6月21日)、家臣・明智光秀に謀反を起こされ、本能寺で自害した。これまで信長の政権は、豊臣秀吉による豊臣政権、徳川家康が開いた江戸幕府への流れをつくった画期的なもので、その政治手法も革新的なものであるとみなされてきた。しかし、近年の歴史学界ではその政策の前時代性が指摘されるようになり、しばしば「中世社会の最終段階」とも評され、その革新性を否定する研究が主流となっている。
検索文書1:  勇功を顕したり」と記されている。家康は、武田信玄を尊敬し、武田氏の遺臣から信玄の戦術や思想を積極的に学んだ。その反面、信長のように身分や序列を無視した徹底的な能力主義をとることはなく、秀吉のように自らのカリスマ性や金、領地を餌に釣って家臣を増やすこともなかった。家康の重臣のほとんどは三河以来の代々仕えてきた家臣たちであった。そのためか、彼らに天下を統一され遅れをとったが、代わりに自身は信頼できる部下だけで周囲を固め、豊臣政権の不備もあって天下人となった。とはいえ、その部下の中には今川氏・武田氏・北条氏等の自身が直接(主導)的には滅ぼしてはいない大名の家臣も含まれているため一種の漁夫の利(統一の際の汚れ役を信長・秀吉が被ってくれた)ともいえる。一方で偉大な先人から学びとり、それを取捨選択しその時流や自分の状況にあう行動をとったことは十分に名君と呼ぶに値するという見方もできる。
検索文書2:  信長は主に技術水準の高い畿内とその周辺の勢力との戦いが多かったのに比べ、秀吉は外域の勢力との戦いが中心だった。そのため大砲を使用する必要性が薄かった。文禄2年(1593年)、肥前・名護屋に参陣した陸奥の大名・南部信直は豊臣軍の軍事力に驚嘆し、やや自虐的に「このたびの御陣な

In [42]:
query = dataset["train"][4]["question"]
docs = retriever.invoke(query)[:3]

# 埋め込みモデルの取得（retrieverの構成に依存）
# FAISS / Chroma などのVectorStoreRetrieverの場合:
embeddings = retriever.vectorstore.embeddings

# ベクトル化
q_vec = np.array(embeddings.embed_query(query))
doc_vecs = np.array(embeddings.embed_documents([d.page_content for d in docs]))

for i, dv in enumerate(doc_vecs):
    sim = cosine_sim(q_vec, dv)
    print(f"検索文書{i} のコサイン類似度: {sim:.4f}")

検索文書0 のコサイン類似度: 0.5605
検索文書1 のコサイン類似度: 0.5356
検索文書2 のコサイン類似度: 0.5270
